# odseki_nazivi ↔ posek_processed ↔ heatmap coverage

Questions:
1. Are all odseki in `odseki_nazivi.csv` also in `posek_processed.csv` (and vice versa)?
2. Are all odseki in `heatmap_past_data.csv` also in `odseki_nazivi.csv`?
3. Can we calculate **relative posek** (heatmap target ÷ povrsina) for the segments we care about?

> Note: `posek_processed` is used for the **detail panel** (m³ per month when you click a segment).
> The **heatmap tile colouring** uses `heatmap_past_data` / `heatmap_future_predictions`.
> `odseki_nazivi` provides `povrsina` (area) needed for the relative posek calculation.

In [1]:
import pandas as pd
from pathlib import Path

BASE = Path("..").resolve()

nazivi  = pd.read_csv(BASE / "data" / "odseki_nazivi.csv")
posek   = pd.read_csv(BASE / "data" / "posek_processed.csv")
heatmap = pd.read_csv(BASE / "data" / "heatmap_past_data.csv")

# Normalise column names and whitespace
nazivi.columns  = nazivi.columns.str.strip()
posek.columns   = posek.columns.str.strip()
heatmap.columns = heatmap.columns.str.strip()

for df, col in [(nazivi, "odsek"), (nazivi, "ggo_naziv"),
                (posek,  "odsek"), (posek,  "ggo"),
                (heatmap,"odsek_id")]:
    df[col] = df[col].astype(str).str.strip()

print(f"odseki_nazivi:    {len(nazivi):>8,} rows")
print(f"posek_processed:  {len(posek):>8,} rows")
print(f"heatmap_past:     {len(heatmap):>8,} rows")
print()
print("odseki_nazivi  columns:", list(nazivi.columns))
print("posek_processed columns:", list(posek.columns))
print("heatmap_past   columns:", list(heatmap.columns))

/tmp/ipykernel_1139021/4015524866.py:7: DtypeWarning: Columns (0: manual) have mixed types. Specify dtype option on import or set low_memory=False.
  posek   = pd.read_csv(BASE / "data" / "posek_processed.csv")


odseki_nazivi:      53,254 rows
posek_processed:   450,061 rows
heatmap_past:     8,225,663 rows

odseki_nazivi  columns: ['ggo_naziv', 'odsek', 'povrsina', 'gge_naziv', 'ke_naziv', 'revir_naziv', 'katgozd_naziv', 'ohranjen_naziv', 'relief_naziv', 'lega_naziv', 'pozar_naziv', 'intgosp_naziv', 'krajime', 'grt1_naziv']
posek_processed columns: ['ggo', 'odsek', 'manual', 'kubikov', 'vrsec', 'posekano', 'ko', 'parcela', 'leto', 'mesec']
heatmap_past   columns: ['ggo', 'odsek_id', 'leto_mesec', 'target']


In [2]:
# Inspect key identifier values to understand formats before comparing
print("=== odseki_nazivi — sample odsek / ggo_naziv ===")
print(nazivi[["ggo_naziv","odsek"]].head(8).to_string(index=False))

print("\n=== posek_processed — sample odsek / ggo ===")
print(posek[["ggo","odsek"]].head(8).to_string(index=False))

print("\n=== heatmap_past — sample odsek_id / ggo ===")
print(heatmap[["ggo","odsek_id"]].head(8).to_string(index=False))

=== odseki_nazivi — sample odsek / ggo_naziv ===
ggo_naziv  odsek
    CELJE  31001
    CELJE  31002
    CELJE  31003
    CELJE  31004
    CELJE  31005
    CELJE 31006A
    CELJE 31006B
    CELJE  31007

=== posek_processed — sample odsek / ggo ===
ggo  odsek
  1 01026A
  1 01076A
  1 01076A
  1 01086A
  1  02141
  1 02163B
  1 02163B
  1 07001B

=== heatmap_past — sample odsek_id / ggo ===
 ggo odsek_id
   1   01003A
   1   01003A
   1   01003A
   1   01003A
   1   01003A
   1   01003A
   1   01003A
   1   01003A


## 1. Unique segments per file

Unique key is `(odsek_id, ggo)` — build a normalised set for each file before comparing.

In [3]:
GGO_CODE_TO_NAZIV = {
    1: 'TOLMIN', 2: 'BLED', 3: 'KRANJ', 4: 'LJUBLJANA', 5: 'POSTOJNA',
    6: 'KOČEVJE', 7: 'NOVO MESTO', 8: 'BREŽICE', 9: 'CELJE',
    10: 'NAZARJE', 11: 'SLOVENJ GRADEC', 12: 'MARIBOR',
    13: 'MURSKA SOBOTA', 14: 'SEŽANA',
}
GGO_NAZIV_TO_CODE = {v: k for k, v in GGO_CODE_TO_NAZIV.items()}

# odseki_nazivi: key is (odsek, ggo_naziv) — spaces removed from odsek
nazivi["odsek_norm"] = nazivi["odsek"].str.replace(' ', '', regex=False)
nazivi["ggo_naziv_norm"] = nazivi["ggo_naziv"].str.strip()
nazivi_segs = set(zip(nazivi["odsek_norm"], nazivi["ggo_naziv_norm"]))

# posek_processed: ggo is a zero-padded integer string ("01"), map to ggo_naziv
# Try both int() and zero-padded string to be safe
def ggo_to_naziv(val):
    try:
        return GGO_CODE_TO_NAZIV.get(int(str(val).lstrip('0') or '0'), None)
    except ValueError:
        return None

posek["ggo_naziv"] = posek["ggo"].apply(ggo_to_naziv)
posek_unknown = posek[posek["ggo_naziv"].isna()]
if len(posek_unknown):
    print(f"WARNING: {len(posek_unknown):,} posek rows with unmapped ggo: {sorted(posek_unknown['ggo'].unique())}")
    posek = posek[posek["ggo_naziv"].notna()].copy()

posek["odsek_norm"] = posek["odsek"].str.replace(' ', '', regex=False)
posek_segs = set(zip(posek["odsek_norm"], posek["ggo_naziv"]))

# heatmap: ggo is an integer
heatmap["ggo_naziv"] = heatmap["ggo"].apply(lambda v: GGO_CODE_TO_NAZIV.get(int(v), None))
heatmap_unknown = heatmap[heatmap["ggo_naziv"].isna()]
if len(heatmap_unknown):
    print(f"WARNING: {len(heatmap_unknown):,} heatmap rows with unmapped ggo: {sorted(heatmap_unknown['ggo'].unique())}")
    heatmap = heatmap[heatmap["ggo_naziv"].notna()].copy()

heatmap["odsek_norm"] = heatmap["odsek_id"].str.replace(' ', '', regex=False)
heatmap_segs = set(zip(heatmap["odsek_norm"], heatmap["ggo_naziv"]))

print(f"Unique (odsek, ggo) segments:")
print(f"  odseki_nazivi:   {len(nazivi_segs):>7,}")
print(f"  posek_processed: {len(posek_segs):>7,}")
print(f"  heatmap_past:    {len(heatmap_segs):>7,}")

Unique (odsek, ggo) segments:
  odseki_nazivi:    53,254
  posek_processed:  34,417
  heatmap_past:     34,417


## 2. odseki_nazivi ↔ posek_processed

In [4]:
in_both_np      = nazivi_segs & posek_segs
only_in_nazivi  = nazivi_segs - posek_segs
only_in_posek   = posek_segs  - nazivi_segs

print(f"{'In both (nazivi ∩ posek):':40s} {len(in_both_np):>7,}")
print(f"{'Only in nazivi (no posek data):':40s} {len(only_in_nazivi):>7,}")
print(f"{'Only in posek (no area/name data):':40s} {len(only_in_posek):>7,}")

# Breakdown by GGO for the gaps
if only_in_nazivi:
    df_miss = pd.DataFrame(sorted(only_in_nazivi), columns=["odsek_id","ggo_naziv"])
    print("\n--- Nazivi segments with no posek data — by GGO ---")
    print(df_miss.groupby("ggo_naziv").size().sort_values(ascending=False).rename("count").to_string())

if only_in_posek:
    df_extra = pd.DataFrame(sorted(only_in_posek), columns=["odsek_id","ggo_naziv"])
    print("\n--- Posek segments with no nazivi entry — by GGO ---")
    print(df_extra.groupby("ggo_naziv").size().sort_values(ascending=False).rename("count").to_string())

In both (nazivi ∩ posek):                 34,358
Only in nazivi (no posek data):           18,896
Only in posek (no area/name data):            59

--- Nazivi segments with no posek data — by GGO ---
ggo_naziv
MARIBOR           3240
CELJE             2930
SEŽANA            2339
MURSKA SOBOTA     1878
TOLMIN            1847
POSTOJNA          1210
KOČEVJE           1147
BREŽICE           1047
LJUBLJANA          832
NOVO MESTO         819
SLOVENJ GRADEC     436
KRANJ              431
BLED               374
NAZARJE            366

--- Posek segments with no nazivi entry — by GGO ---
ggo_naziv
NAZARJE           36
SEŽANA            13
BREŽICE            6
MURSKA SOBOTA      2
SLOVENJ GRADEC     2


## 3. heatmap_past ↔ odseki_nazivi — can we get povrsina for every heatmap segment?

In [5]:
in_both_hn      = heatmap_segs & nazivi_segs
only_in_heatmap = heatmap_segs - nazivi_segs
only_in_nazivi2 = nazivi_segs  - heatmap_segs

print(f"{'In both (heatmap ∩ nazivi):':40s} {len(in_both_hn):>7,}")
print(f"{'Heatmap segments missing povrsina:':40s} {len(only_in_heatmap):>7,}  ← cannot do relative posek")
print(f"{'Nazivi segments not in heatmap:':40s} {len(only_in_nazivi2):>7,}")

coverage_pct = len(in_both_hn) / len(heatmap_segs) * 100
print(f"\nPovrsina coverage for heatmap segments: {coverage_pct:.1f}%")

if only_in_heatmap:
    df_miss2 = pd.DataFrame(sorted(only_in_heatmap), columns=["odsek_id","ggo_naziv"])
    print("\n--- Heatmap segments with no povrsina — by GGO ---")
    print(df_miss2.groupby("ggo_naziv").size().sort_values(ascending=False).rename("count").to_string())

In both (heatmap ∩ nazivi):               34,358
Heatmap segments missing povrsina:            59  ← cannot do relative posek
Nazivi segments not in heatmap:           18,896

Povrsina coverage for heatmap segments: 99.8%

--- Heatmap segments with no povrsina — by GGO ---
ggo_naziv
NAZARJE           36
SEŽANA            13
BREŽICE            6
MURSKA SOBOTA      2
SLOVENJ GRADEC     2


## 4. povrsina sanity check — unit and value distribution

In [6]:
nazivi["povrsina"] = pd.to_numeric(nazivi["povrsina"], errors="coerce")

null_area = nazivi["povrsina"].isna().sum()
zero_area = (nazivi["povrsina"] == 0).sum()

print(f"povrsina — null values:  {null_area:,}")
print(f"povrsina — zero values:  {zero_area:,}")
print(f"povrsina — valid values: {nazivi['povrsina'].notna().sum():,}")
print()
print(nazivi["povrsina"].describe().round(2))
print()
# A typical Slovenian gozd odsek is a few to a few hundred ha.
# If values are in ha, max should be < ~10,000; if m² they'd be much larger.
print(f"Min: {nazivi['povrsina'].min():.2f}  Max: {nazivi['povrsina'].max():.2f}  Median: {nazivi['povrsina'].median():.2f}")
print("(If values are in ha, median should be roughly 10–200 ha for forest segments)")

povrsina — null values:  0
povrsina — zero values:  2
povrsina — valid values: 53,254

count    53254.00
mean        22.11
std         20.36
min          0.00
25%          8.56
50%         17.44
75%         29.85
max        314.42
Name: povrsina, dtype: float64

Min: 0.00  Max: 314.42  Median: 17.44
(If values are in ha, median should be roughly 10–200 ha for forest segments)


## 5. Summary — can we implement relative posek?

In [7]:
print("=" * 60)
print("SUMMARY")
print("=" * 60)

print(f"\n1. odseki_nazivi ↔ posek_processed")
print(f"   Overlap: {len(in_both_np):,} / {len(nazivi_segs):,} nazivi segments ({len(in_both_np)/len(nazivi_segs)*100:.1f}%)")
print(f"   {len(only_in_nazivi):,} nazivi segments have no posek records")
print(f"   {len(only_in_posek):,} posek segments have no entry in nazivi")

print(f"\n2. heatmap ↔ odseki_nazivi  (needed for relative posek)")
print(f"   {len(in_both_hn):,} / {len(heatmap_segs):,} heatmap segments have a povrsina ({coverage_pct:.1f}%)")
print(f"   {len(only_in_heatmap):,} heatmap segments would fall back to absolute value")

print(f"\n3. povrsina quality")
print(f"   Null/zero area values: {null_area + zero_area:,} out of {len(nazivi):,}")

print(f"\n→ Relative posek (target / povrsina) is feasible for {coverage_pct:.0f}% of heatmap segments.")
if only_in_heatmap:
    print(f"  The remaining {len(only_in_heatmap):,} would need a fallback (e.g. absolute bucket or greyed out).")
else:
    print(f"  Full coverage — no fallback needed.")

SUMMARY

1. odseki_nazivi ↔ posek_processed
   Overlap: 34,358 / 53,254 nazivi segments (64.5%)
   18,896 nazivi segments have no posek records
   59 posek segments have no entry in nazivi

2. heatmap ↔ odseki_nazivi  (needed for relative posek)
   34,358 / 34,417 heatmap segments have a povrsina (99.8%)
   59 heatmap segments would fall back to absolute value

3. povrsina quality
   Null/zero area values: 2 out of 53,254

→ Relative posek (target / povrsina) is feasible for 100% of heatmap segments.
  The remaining 59 would need a fallback (e.g. absolute bucket or greyed out).
